In [11]:
from cropspectai import config
from cropspectai.data_analysis import preprocessing
from cropspectai.modeling.train import train_model

import pandas as pd

from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor


# --------------------------------------------------
# Load data
# --------------------------------------------------

spectral_length = "as7343" # "full" or "1000" or "as7343"
target = "cab" # "cab" or None
indices = None # "all" or "established" or None

if spectral_length == "as7343":
    if target == None:
        train_path = (
            config.PROCESSED_PARQUET_DATA_SPLITS_PATH
            / "all"
            / "train.parquet"
        )
    elif target == "cab":
        if indices == "all": # all engineered features
            train_path = (
                        config.PROCESSED_PARQUET_DATA_SPLITS_PATH
                        / target
                        / "train_all_features.parquet"
                    )
        elif indices == "established": # only established new features
            train_path = (
                        config.PROCESSED_PARQUET_DATA_SPLITS_PATH
                        / target
                        / "train_established_features.parquet"
                    ) 
        elif indices == None: # no engineered features
            train_path = (
                        config.PROCESSED_PARQUET_DATA_SPLITS_PATH
                        / target
                        / "train.parquet"
                    )       
elif spectral_length == "full" or "1000":
    train_path = (
        config.RAW_PARQUET_DATA_SPLITS_PATH
        / "train.parquet"
    )

df = pd.read_parquet(train_path)
print("Shape of imported data: ", df.shape)

Shape of imported data:  (1607, 13)


In [12]:
# --------------------------------------------------
# Get feature and target column names
# --------------------------------------------------

X_all, y_all = preprocessing.split_dataset_into_X_y(df, target)

targets = y_all.columns.tolist()

print("Targets:")
print(targets)

Targets:
['cab']


## Hyperspectra up to 1000nm only

In [13]:
if spectral_length == "1000":
    print(X_all.shape)
    X_1000 = X_all.loc[:, "400":"1000"].copy()
    print(X_1000.shape)
    print(y_all.shape)

    df = pd.concat([X_1000, y_all], axis = 1).copy()
    print(df.shape)

In [14]:
# --------------------------------------------------
# Store results
# --------------------------------------------------

results = {}


# --------------------------------------------------
# Run XGBoost independently for every target
# --------------------------------------------------

for target in targets:

    print("\n" + "=" * 60)
    print(f"TARGET: {target}")
    print("=" * 60)

    # IMPORTANT:
    # Start from the original dataframe for every target
    df_target = preprocessing.drop_duplicates(
        df,
        target
    ).copy()

    # Split into features and targets AFTER target-specific
    # duplicate handling
    X, y = preprocessing.split_dataset_into_X_y(df_target, target)

    # # Select only current target
    # y = y[target].copy()

    # Train / validation split
    X_train, X_val, y_train, y_val = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
    )

    # Create a fresh model for every target
    model_xgb = XGBRegressor(
        n_estimators=1000,
        max_depth=6,
        learning_rate=0.01,
        subsample=0.8,
        colsample_bytree=1.0,

        tree_method="hist",

        objective="reg:squarederror",
        eval_metric="rmse",

        n_jobs=-1,
        random_state=42,
    )

    # Train
    pipeline_xgb, metrics = train_model(
        model_xgb,
        X_train,
        y_train,
        X_val,
        y_val,
        scale=False,
    )

    metrics["Samples"] = len(df_target)
    results[target] = metrics


TARGET: cab
(Before) Shape of dataframe: (1607, 13)
Dropping duplicate rows
Duplicate rows: 1
(After) Shape of dataframe: (1606, 13)
Checking for rows without labels
Rows with no labels: 0
(After) Shape of dataframe: (1606, 13)


In [15]:
# --------------------------------------------------
# Create results table
# --------------------------------------------------

results_df = pd.DataFrame(results)
print("\nXGBoost Results for each target")
print(results_df)


XGBoost Results for each target
                          cab
MAE (train)          2.861355
MAPE (train) [%]     8.948772
MAE (test)           6.843195
MAPE (test) [%]     20.022470
R² (test)            0.534232
Samples           1606.000000


In [16]:
file_name = f"xgboost_results_{target}_{spectral_length}-{indices}.csv"
output_path = config.REPORTS_DIR / file_name

results_df.to_csv(
    output_path,
    index=True,
)

## compare results

In [ ]:
# file_name_1000 = f"xgboost_results_1000.csv"
# file_name_full = f"xgboost_results_full.csv"
# file_name_as7343 = f"xgboost_results_as7343.csv"

# results_1000 = pd.read_csv(config.REPORTS_DIR / file_name_1000, index_col=0)
# results_full = pd.read_csv(config.REPORTS_DIR / file_name_full, index_col=0)
# results_as7343 = pd.read_csv(config.REPORTS_DIR / file_name_as7343, index_col=0)

# print("as7343")
# print(results_as7343)

# print("\n full")
# print(results_full)

# print("\n 1000")
# print(results_1000)




# full_vs_as7343 = abs(results_full - results_as7343) / results_full *100
# thousand_vs_as7343 = abs(results_1000 - results_as7343) / results_1000 * 1000
# full_vs_thousand = abs(results_full - results_1000) / results_full *100

# print("Full vs AS7343")
# print(full_vs_as7343)

# print("\n1000 vs AS7343")
# print(thousand_vs_as7343)

# print("\nFull vs 1000")
# print(full_vs_thousand)

- full length hyperspectra gives better R2 score for all targets except CAB
- 1000nm cutoff hyperspectra gives consistently lower R2 score than full except for CAB
- as7343 does poor at all targets and consistently lower than 1000nm 
 - but CAB score remains on par with 1000nm
 - more importantly does better than even full

Conclusion:
- CAB is better predicted with data within 1000nm range
- as7343 is better capable at predicting CAB than all other targets